In [2]:
from google.colab import drive
import os

drive.mount('/content/drive')
if not os.path.exists('/content/drive/MyDrive/Recipes-Recommendation-System'):
    os.makedirs("/content/drive/MyDrive/Recipes-Recommendation-System")
    %cd /content/drive/MyDrive/Recipes-Recommendation-System
    !git clone https://github.com/soheil-housni/Recipes-Recommendation-System.git \
        /content/drive/MyDrive/Recipes-Recommendation-System

else:
    %cd /content/drive/MyDrive/Recipes-Recommendation-System
    !git pull origin main

print(os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/Recipes-Recommendation-System
From https://github.com/soheil-housni/Recipes-Recommendation-System
 * branch            main       -> FETCH_HEAD
Already up to date.
/content/drive/MyDrive/Recipes-Recommendation-System


In [3]:
!pip install mlflow
!pip install loguru
!pip install optuna

In [1]:
import pandas as pd
import numpy as np
import ast
import numpy as np
import torch
import mmh3
from transformers import AutoTokenizer,DistilBertModel
import itertools
import pyarrow.parquet as pq
from torch.utils.data import DataLoader
from sklearn.preprocessing import StandardScaler
import mlflow
from modules.data_preparation import BERTCreationDataset, CreationDataset,CollateFunction,CollateFunctionInferenceRecipes,CollateFunctionInferenceUsers,BertCollateFunction
from modules.extraction import BERTEmbeddingsExtractor,EncodedHashedEmbeddings
from modules.inference import InferencePreprocessingRecipes,InferencePreprocessingUsers,RecipesRecommender
from modules.model import ContrastiveMSELoss,RecommendationModel,OptunaFunction,Test,Train
from modules.preprocessing import AfterSplitPreprocessingUsersFeatures,AfterSplitPreprocessingRecipesFeatures,BeforeSplitPreprocessingRecipesFeatures,BeforeSplitPreprocessingUsersFeatures,Scaler, ScalerInferenceRecipes, ScalerInferenceUsers,split_df
from modules.recipes_extraction import RecipesEmbeddingsExtractor,creation_recipes_set
from modules.utils import seed_worker,set_seed,load
import optuna
from loguru import logger
import gc
import os
import plotly.io as pio
import joblib
from mlflow.tracking import MlflowClient
pio.renderers.default = "vscode"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [2]:
seed=42
set_seed(seed=seed)

In [3]:
mlflow.set_tracking_uri("file:./mlflow_runs/mlruns")
mlflow.set_experiment("Recipes_Recommendation_System_model_training")

c:\Users\sh032\anaconda3\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning:

The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.



<Experiment: artifact_location=('file:///C:/Users/sh032/OneDrive - HEC Montréal/Documents/Master 1/Recipe '
 'recommendation system/mlflow_runs/mlruns/896236271089450697'), creation_time=1776517499453, experiment_id='896236271089450697', last_update_time=1776517499453, lifecycle_stage='active', name='Recipes_Recommendation_System_model_training', tags={}, workspace='default'>

In [4]:
client=MlflowClient()

## Creation of the processed dataframe

In [3]:
interactions_train_df=pd.read_csv("data/interactions_train.csv")
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_users_df=pd.read_csv("data/PP_users.csv")
RAW_interactions_df=pd.read_csv("data/RAW_interactions.csv")
RAW_recipes_df=pd.read_csv("data/RAW_recipes.csv")


In [4]:
interactions_train_df_filtered=interactions_train_df.sample(n=20000,replace=False,random_state=seed)

In [5]:
print(interactions_train_df_filtered.shape)


(20000, 6)


In [6]:
RAW_recipes_df.rename(columns={"id":"recipe_id"},inplace=True)
PP_recipes_df.rename(columns={"techniques":"techniques_recipes"},inplace=True)
PP_users_df.rename(columns={"techniques":"techniques_users"},inplace=True)

In [7]:
print("interactions_train_df_filtered columns:")
print(interactions_train_df_filtered.columns)
print("------------------------------------------------------------------")
print("PP_recipes_df columns:")
print(PP_recipes_df.columns)
print("-------------------------------------------------------------------")
print("PP_users_df columns:")
print(PP_users_df.columns)
print("----------------------------------------------------------------------")
print("RAW_interactions_df columns:")
print(RAW_interactions_df.columns)
print("------------------------------------------------------------------------")
print("RAW_recipes_df columns:")
print(RAW_recipes_df.columns)

interactions_train_df_filtered columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'u', 'i'], dtype='object')
------------------------------------------------------------------
PP_recipes_df columns:
Index(['id', 'i', 'name_tokens', 'ingredient_tokens', 'steps_tokens',
       'techniques_recipes', 'calorie_level', 'ingredient_ids'],
      dtype='object')
-------------------------------------------------------------------
PP_users_df columns:
Index(['u', 'techniques_users', 'items', 'n_items', 'ratings', 'n_ratings'], dtype='object')
----------------------------------------------------------------------
RAW_interactions_df columns:
Index(['user_id', 'recipe_id', 'date', 'rating', 'review'], dtype='object')
------------------------------------------------------------------------
RAW_recipes_df columns:
Index(['name', 'recipe_id', 'minutes', 'contributor_id', 'submitted', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'ingredients',
       'n_ingredients'],
      dty

In [8]:
full_df=interactions_train_df_filtered.merge(PP_recipes_df,on="i",how="left",suffixes=(None,"_1")).merge(PP_users_df,on="u",how="left",suffixes=(None,"_2")).merge(RAW_interactions_df,on=["user_id","recipe_id"],how="left",suffixes=(None,"_3")).merge(RAW_recipes_df,on=["recipe_id"],how="left",suffixes=(None,"_3"))

In [9]:
full_df=full_df.drop(columns=["review",'rating_3',"id","ingredients","submitted","contributor_id","date_3","date","steps_tokens","ingredient_tokens","name_tokens","u"],axis=1)
full_df.to_parquet(path="./modules/dataframes/full_df.parquet")

In [10]:
full_parquet_file=pq.ParquetFile("./modules/dataframes/full_df.parquet")
full_df=load(full_parquet_file)

In [11]:
print(f"Columns of the full dataframe:")
print(full_df.columns)
print("----------------------------------------")
print(f"Shape of the full dataframe:")
print(full_df.shape)
print("----------------------------------------")
print(f"Columns of the filtered interaction dataframe:")
print(interactions_train_df_filtered.shape)
print("----------------------------------------")
print(f"Head of the full  dataframe:")
display(full_df.head())
print("----------------------------------------")
for col in full_df.columns:
    print(f"Unique values of {col} :")
    print(full_df[col].unique())
    print("----------------------------------------")
for col in full_df.columns:
    print(f"The type of {col} is")
    print(full_df[col].apply(type).unique())
    print("---------------------------------")

Columns of the full dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients'],
      dtype='object')
----------------------------------------
Shape of the full dataframe:
(20000, 20)
----------------------------------------
Columns of the filtered interaction dataframe:
(20000, 6)
----------------------------------------
Head of the full  dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,ratings,n_ratings,name,minutes,tags,nutrition,n_steps,steps,description,n_ingredients
0,231160,124810,5.0,170682,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4574, 1910, 6906, 2499, 63, 332, 6270, 7470, ...","[3, 0, 0, 0, 1, 0, 0, 0, 0, 2, 1, 0, 0, 0, 0, ...","[8911, 20238, 117521, 79316, 154563, 34302, 17...",7,"[5.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0]",7,huckleberry or blueberry coffee cake,75,"['time-to-make', 'course', 'main-ingredient', ...","[174.1, 4.0, 92.0, 8.0, 7.0, 3.0, 11.0]",11,['beat margarine and cream cheese at medium sp...,"cooking light published this in their book, fi...",10
1,142629,31342,5.0,120865,"[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",1,"[1168, 1284, 8021, 2499, 330, 6270, 3217, 2318...","[6, 0, 0, 3, 1, 0, 0, 0, 0, 6, 0, 0, 0, 0, 0, ...","[122019, 150060, 117906, 82202, 119046, 141911...",10,"[4.0, 5.0, 5.0, 5.0, 4.0, 5.0, 3.0, 5.0, 4.0, ...",10,blender quiche or whatever you have in your ...,65,"['weeknight', 'time-to-make', 'course', 'main-...","[308.4, 37.0, 8.0, 20.0, 21.0, 41.0, 3.0]",10,"['preheat oven to 350f', 'generously grease a ...",the great part is you can use whatever leftove...,12
2,822358,441841,4.0,23142,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[1648, 1706, 3347, 5298, 3044, 4623, 2148, 6270]","[1, 1, 0, 2, 3, 0, 0, 0, 1, 5, 0, 1, 0, 0, 0, ...","[29761, 87944, 72370, 139097, 39470, 37755, 23...",17,"[4.0, 5.0, 5.0, 5.0, 4.0, 4.0, 4.0, 5.0, 5.0, ...",17,sweet orange coleslaw,10,"['15-minutes-or-less', 'time-to-make', 'course...","[290.1, 21.0, 71.0, 14.0, 7.0, 8.0, 13.0]",6,"['chop the apples', 'in a small bowl mix the o...",this recipe came from one of my mom's cookbook...,8
3,655579,211110,4.0,156541,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[7031, 5006, 339, 3203, 5975, 6270, 590]","[10, 0, 0, 5, 11, 0, 0, 0, 0, 7, 1, 3, 0, 0, 1...","[34492, 71279, 73805, 68721, 114795, 35164, 33...",28,"[2.0, 3.0, 4.0, 2.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",28,wilted greens with garlic and balsamic vinegar,15,"['15-minutes-or-less', 'time-to-make', 'course...","[77.5, 10.0, 5.0, 5.0, 2.0, 4.0, 1.0]",5,"['cut the greens into strips 1 inch wide', 'in...",this simple but delicious side dish can be mad...,7
4,35140,114469,5.0,30839,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",2,"[1240, 6270, 5006, 1257, 2683, 5168, 800, 2461...","[176, 4, 2, 55, 72, 0, 1, 7, 4, 141, 13, 15, 0...","[121722, 36691, 151293, 154016, 1205, 12802, 1...",364,"[5.0, 5.0, 0.0, 5.0, 5.0, 5.0, 5.0, 5.0, 5.0, ...",364,chicken in the limelight,75,"['time-to-make', 'course', 'main-ingredient', ...","[645.9, 69.0, 11.0, 22.0, 80.0, 54.0, 4.0]",8,"['preheat oven to 350', 'grate lime peel and s...",very tender chicken soaked with flavor,9


----------------------------------------
Unique values of user_id :
[231160 142629 822358 ... 241982 164867 137854]
----------------------------------------
Unique values of recipe_id :
[124810  31342 441841 ...  98278 274099 283283]
----------------------------------------
Unique values of rating :
[5. 4. 3. 2. 0. 1.]
----------------------------------------
Unique values of i :
[170682 120865  23142 ... 108495 128044  25867]
----------------------------------------
Unique values of techniques_recipes :
['[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[1, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]'
 '[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [12]:
PP_users_df["len_items"]=PP_users_df["items"].str.count(r",")+1
PP_recipes_df["len_ingredients"]=PP_recipes_df["ingredient_ids"].str.count(r",")+1
max_len_ingredients=PP_recipes_df["len_ingredients"].max()
max_len_items=PP_users_df["len_items"].max()
PP_users_df.drop(columns=["len_items"],inplace=True)
PP_recipes_df.drop(columns=["len_ingredients"],inplace=True)

In [13]:
before_split_users_processor=BeforeSplitPreprocessingUsersFeatures(full_df,max_len_items=max_len_items)
full_df_processed=before_split_users_processor.preprocessing()

In [14]:
before_split_recipes_processor=BeforeSplitPreprocessingRecipesFeatures(full_df_processed,max_len_ingredients=max_len_ingredients)
full_df_processed=before_split_recipes_processor.preprocessing()

In [15]:
full_df_processed["rating_scaled"]=full_df_processed["rating"]/5
full_df_processed["ratings_scaled"]=full_df_processed["ratings"].apply(lambda x: list(map(lambda x: x/5,x)))

In [16]:
PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(lambda x: int(x)+1,ast.literal_eval(x))))
all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))

In [20]:
def mapping_list(x:list,mapping_serie:pd.Series):
    return [mapping_serie.loc[i] if i in mapping_serie.index else 0 for i in x]

In [21]:
full_df_processed["ingredient_ids_continuous"]=full_df_processed["ingredient_ids"].apply(lambda x : mapping_list(x,ingredient_continuous_ids))

In [22]:
full_df_processed["full_text"]=full_df_processed["name"]+" [SEP] "+full_df_processed["description"]+" [SEP] "+full_df_processed["steps"]+" [SEP] "+full_df_processed["tags"]

In [ ]:
#full_df_processed.to_parquet(path="./modules/dataframes/full_df_processed.parquet")

In [ ]:
#full_processed_parquet_file = pq.ParquetFile("./modules/dataframes/full_df_processed.parquet")
#full_df_processed=load(full_processed_parquet_file)

In [ ]:
#df_text=full_df_processed["full_text"]

In [ ]:
#distilbert_model=DistilBertModel.from_pretrained("distilbert-base-uncased")
#tokenizer=AutoTokenizer.from_pretrained("distilbert-base-uncased")

In [ ]:
#bert_dataset=BERTCreationDataset(df_text)

In [ ]:
#generator=torch.Generator()
#generator.manual_seed(seed)
#bert_collate_function_object=BertCollateFunction(tokenizer=tokenizer)
#bert_dataloader=DataLoader(dataset=bert_dataset,batch_size=32,shuffle=False,collate_fn=bert_collate_function_object.collate_fn,worker_init_fn=seed_worker,generator=generator)

In [ ]:
#device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
#bert_extractor=BERTEmbeddingsExtractor(bert_model=distilbert_model,device=device)
#bert_extractor.get_bert_embeddings(dataloader=bert_dataloader,path="./modules/BERT_embeddings")

In [23]:
all_cls_embeddings=torch.load("./modules/BERT_embeddings/all_cls_embeddings.pt")
all_mean_embeddings=torch.load("./modules/BERT_embeddings/all_mean_embeddings.pt")
print(all_cls_embeddings.shape)
print(all_mean_embeddings.shape)

torch.Size([20000, 768])
torch.Size([20000, 768])


In [24]:
train_df,val_df,test_df,train_cls_embeddings,val_cls_embeddings,test_cls_embeddings,train_mean_embeddings,val_mean_embeddings,test_mean_embeddings=split_df(full_df_processed,cls_embeddings=all_cls_embeddings,mean_embeddings=all_mean_embeddings,random_state=seed)

In [25]:
after_split_preprocessor_users=AfterSplitPreprocessingUsersFeatures(train_df)
train_df_processed=after_split_preprocessor_users.preprocessing(train_df)
val_df_processed=after_split_preprocessor_users.preprocessing(val_df)
test_df_processed=after_split_preprocessor_users.preprocessing(test_df)

In [26]:
after_split_preprocessor_recipes=AfterSplitPreprocessingRecipesFeatures(train_df)
train_df_processed=after_split_preprocessor_recipes.preprocessing(train_df_processed)
val_df_processed=after_split_preprocessor_recipes.preprocessing(val_df_processed)
test_df_processed=after_split_preprocessor_recipes.preprocessing(test_df_processed)

In [27]:
scaler_users=StandardScaler()
scaler_recipes=StandardScaler()
scale_processor=Scaler(scaler_users,scaler_recipes)
train_df_processed,val_df_processed,test_df_processed=scale_processor.scale(train_df_processed,val_df_processed,test_df_processed)

In [28]:
joblib.dump(scale_processor.scaler_users, "./modules/scalers/scaler_users.pkl")
joblib.dump(scale_processor.scaler_recipes, "./modules/scalers/scaler_recipes.pkl")

['./modules/scalers/scaler_recipes.pkl']

In [29]:
scaler_users=joblib.load("./modules/scalers/scaler_users.pkl")
scaler_recipes=joblib.load("./modules/scalers/scaler_recipes.pkl")

In [30]:
train_df.to_parquet(path="./modules/dataframes/train_df.parquet")

train_df_processed.to_parquet(path="./modules/dataframes/train_df_processed.parquet")
val_df_processed.to_parquet(path="./modules/dataframes/val_df_processed.parquet")
test_df_processed.to_parquet(path="./modules/dataframes/test_df_processed.parquet")

torch.save(train_cls_embeddings,f"./modules/BERT_embeddings/train_cls_embeddings.pt")
torch.save(val_cls_embeddings,f"./modules/BERT_embeddings/val_cls_embeddings.pt")
torch.save(test_cls_embeddings,f"./modules/BERT_embeddings/test_cls_embeddings.pt")

torch.save(train_mean_embeddings,f"./modules/BERT_embeddings/train_mean_embeddings.pt")
torch.save(val_mean_embeddings,f"./modules/BERT_embeddings/val_mean_embeddings.pt")
torch.save(test_mean_embeddings,f"./modules/BERT_embeddings/test_mean_embeddings.pt")


## Load of the dataframes

In [5]:
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_users_df=pd.read_csv("data/PP_users.csv")

full_parquet_file=pq.ParquetFile("./modules/dataframes/full_df.parquet")
full_processed_parquet_file=pq.ParquetFile("./modules/dataframes/full_df_processed.parquet")
train_parquet_file=pq.ParquetFile("./modules/dataframes/train_df.parquet")
train_processed_parquet_file = pq.ParquetFile("./modules/dataframes/train_df_processed.parquet")
val_processed_parquet_file= pq.ParquetFile("./modules/dataframes/val_df_processed.parquet")
test_processed_parquet_file= pq.ParquetFile("./modules/dataframes/test_df_processed.parquet")

full_df=load(full_parquet_file)
full_df_processed=load(full_processed_parquet_file)
train_df=load(train_parquet_file)
train_df_processed =load(train_processed_parquet_file)
val_df_processed=load(val_processed_parquet_file)
test_df_processed=load(test_processed_parquet_file)

train_cls_embeddings=torch.load(f"./modules/BERT_embeddings/train_cls_embeddings.pt")
val_cls_embeddings=torch.load(f"./modules/BERT_embeddings/val_cls_embeddings.pt")
test_cls_embeddings=torch.load(f"./modules/BERT_embeddings/test_cls_embeddings.pt")

train_mean_embeddings=torch.load(f"./modules/BERT_embeddings/train_mean_embeddings.pt")
val_mean_embeddings=torch.load(f"./modules/BERT_embeddings/val_mean_embeddings.pt")
test_mean_embeddings=torch.load(f"./modules/BERT_embeddings/test_mean_embeddings.pt")

all_cls_embeddings=torch.load("./modules/BERT_embeddings/all_cls_embeddings.pt")
all_mean_embeddings=torch.load("./modules/BERT_embeddings/all_mean_embeddings.pt")

In [8]:
print(f"Columns of the full processed dataframe:")
print(train_df_processed.columns)
print("----------------------------------------")
print(f"Shape of the full processed dataframe:")
print(train_df_processed.shape)
print("----------------------------------------")
print(f"Head of the full  processed dataframe:")
display(train_df_processed.head())
print("----------------------------------------")
for col in train_df_processed.columns:
    print(f"Exemple of {col} :")
    print(train_df_processed.iloc[0][col])
    print("----------------------------------------")
for col in train_df_processed.columns:
    print(f"The type of {col} is")
    print(train_df_processed[col].apply(type).unique())
    print("---------------------------------")

Columns of the full processed dataframe:
Index(['user_id', 'recipe_id', 'rating', 'i', 'techniques_recipes',
       'calorie_level', 'ingredient_ids', 'techniques_users', 'items',
       'n_items', 'ratings', 'n_ratings', 'name', 'minutes', 'tags',
       'nutrition', 'n_steps', 'steps', 'description', 'n_ingredients',
       'rating_scaled', 'ratings_scaled', 'ingredient_ids_continuous',
       'full_text', 'n_items_scaled', 'n_ratings_scaled',
       'calorie_level_scaled', 'minutes_scaled', 'n_steps_scaled',
       'n_ingredients_scaled'],
      dtype='object')
----------------------------------------
Shape of the full processed dataframe:
(14000, 30)
----------------------------------------
Head of the full  processed dataframe:


,user_id,recipe_id,rating,i,techniques_recipes,calorie_level,ingredient_ids,techniques_users,items,n_items,...,rating_scaled,ratings_scaled,ingredient_ids_continuous,full_text,n_items_scaled,n_ratings_scaled,calorie_level_scaled,minutes_scaled,n_steps_scaled,n_ingredients_scaled
0,175728,11770,5,109126,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0,"[4624, 6655, 217, 6927, 6336, 4864, 0, 0, 0, 0...","[87, 2, 0, 27, 42, 1, 1, 7, 0, 80, 3, 3, 1, 0,...","[154350, 54820, 77571, 45268, 124558, 38966, 1...",202,...,1.0,"[1.0, 1.0, 0.8, 1.0, 0.8, 1.0, 0.8, 0.6, 0.8, ...","[4609, 6629, 215, 6900, 6313, 4847, 0, 0, 0, 0...",walnut brewery asiago cheese dip [SEP] this a ...,-0.386555,-0.386555,-1.086207,-0.156327,0.050739,-0.923525
1,50510,350360,4,53777,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",2,"[5314, 5007, 3204, 5011, 7214, 589, 2595, 2857...","[127, 1, 2, 91, 191, 0, 0, 14, 3, 176, 10, 12,...","[59108, 117303, 16669, 165155, 78915, 115100, ...",580,...,0.8,"[0.8, 1.0, 1.0, 0.8, 1.0, 0.8, 0.8, 1.0, 1.0, ...","[5295, 4990, 3195, 4994, 7187, 586, 2588, 2849...",nif s penne with feta [SEP] a delicious and fr...,0.004451,0.004451,1.461012,-0.142037,-0.715266,0.643517
2,9749,13099,4,13490,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[151, 1125, 5299, 1910, 4764, 156, 6907, 6277,...","[44, 1, 1, 23, 33, 0, 1, 5, 2, 35, 10, 5, 0, 0...","[75852, 104812, 129769, 99788, 101820, 166815,...",144,...,0.8,"[0.8, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 0.8, 1.0, ...","[149, 1119, 5281, 1903, 4748, 154, 6880, 6254,...",mom s waldorf salad [SEP] this is my mother's ...,-0.446551,-0.446551,0.187403,-0.184908,-0.970601,-0.296708
3,161283,56695,0,115884,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",1,"[2731, 3485, 2405, 2500, 6271, 3487, 64, 4054,...","[33, 1, 0, 10, 12, 0, 0, 5, 0, 30, 6, 3, 0, 0,...","[130597, 16122, 49703, 152649, 103416, 36541, ...",92,...,0.0,"[1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, 1.0, ...","[2723, 3476, 2398, 2493, 6248, 3478, 63, 4041,...",the very best salisbury steak [SEP] this is ou...,-0.500340,-0.500340,0.187403,-0.099165,-0.204596,0.643517
4,463436,286054,4,60799,"[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0,"[4718, 7999, 3045, 3830, 0, 0, 0, 0, 0, 0, 0, ...","[111, 1, 2, 69, 82, 0, 1, 9, 1, 114, 17, 20, 1...","[32365, 145711, 157654, 141020, 35883, 67138, ...",395,...,0.8,"[1.0, 1.0, 1.0, 1.0, 1.0, 0.8, 1.0, 1.0, 1.0, ...","[4702, 7969, 3036, 3820, 0, 0, 0, 0, 0, 0, 0, ...",o j yogurt shake [SEP] this is a yummy and ...,-0.186914,-0.186914,-1.086207,-0.227780,-0.970601,-1.550341


----------------------------------------
Exemple of user_id :
175728
----------------------------------------
Exemple of recipe_id :
11770
----------------------------------------
Exemple of rating :
5
----------------------------------------
Exemple of i :
109126
----------------------------------------
Exemple of techniques_recipes :
[1 0 0 0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 1 0 0 0]
----------------------------------------
Exemple of calorie_level :
0
----------------------------------------
Exemple of ingredient_ids :
[4624 6655  217 6927 6336 4864    0    0    0    0    0    0    0    0
    0    0    0    0    0    0]
----------------------------------------
Exemple of techniques_users :
[87  2  0 27 42  1  1  7  0 80  3  3  1  0  3  0 40  0  0  5 20  8  2  8
  9  0  4  4 43  4  1  1  1 58  0  3 19  6 14  1  1  6 29 35  5  1 20  3
  0  5  2  3  0 18  5 18 10 20]
----------------------------------------
Exemple of 

## Model

In [9]:
train_dataset=CreationDataset(train_df_processed,cls_embeddings=train_cls_embeddings,mean_embeddings=train_mean_embeddings)
val_dataset=CreationDataset(val_df_processed,cls_embeddings=val_cls_embeddings,mean_embeddings=val_mean_embeddings)

In [10]:
train_dataset.__getitem__(10)

{'user_id': np.int64(440300),
 'recipe_id': np.int64(5013),
 'rating': np.int64(5),
 'i': np.int64(109198),
 'techniques_recipes': array([0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]),
 'calorie_level': np.int64(1),
 'ingredient_ids': array([  64, 5007, 2500, 7169, 7656, 7957, 6907,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0], dtype=int32),
 'techniques_users': array([5, 0, 0, 0, 2, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 1, 1,
        0, 0, 1, 0, 0, 1, 2, 0, 0, 0, 0, 2, 0, 0, 1, 0, 1, 0, 0, 1, 1, 2,
        0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 2]),
 'items': array([ 18676,  75351, 109198, ...,      0,      0,      0], dtype=int32),
 'n_items': np.int64(10),
 'ratings': array([2, 5, 5, ..., 0, 0, 0], dtype=int32),
 'n_ratings': np.int64(10),
 'name': 'chicago style deep dish pizza crust',
 'minu

In [11]:
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunction()
collate_fn=collate_function_object.collate_fn
train_dataloader=DataLoader(train_dataset,batch_size=64,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)
val_dataloader=DataLoader(val_dataset,batch_size=64,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)

In [12]:
batch=next(iter(train_dataloader))
for key in list(batch.keys()):
    print(key)
    print(f"The shape of batch {key} is :")
    print(batch[key].shape)
    print("-------------------")

user_id
The shape of batch user_id is :
torch.Size([64, 1])
-------------------
recipe_id
The shape of batch recipe_id is :
torch.Size([64, 1])
-------------------
rating_scaled
The shape of batch rating_scaled is :
torch.Size([64, 1])
-------------------
i
The shape of batch i is :
torch.Size([64, 1])
-------------------
technique_recipes
The shape of batch technique_recipes is :
torch.Size([64, 58])
-------------------
calorie_level_scaled
The shape of batch calorie_level_scaled is :
torch.Size([64, 1])
-------------------
techniques_users
The shape of batch techniques_users is :
torch.Size([64, 58])
-------------------
items
The shape of batch items is :
torch.Size([64, 6437])
-------------------
n_items_scaled
The shape of batch n_items_scaled is :
torch.Size([64, 1])
-------------------
ratings_scaled
The shape of batch ratings_scaled is :
torch.Size([64, 6437])
-------------------
n_ratings_scaled
The shape of batch n_ratings_scaled is :
torch.Size([64, 1])
-------------------
mi

In [ ]:
#PP_recipes_df["ingredient_ids_transformed"]=PP_recipes_df["ingredient_ids"].apply(lambda x: list(map(int,ast.literal_eval(x))))
#all_ingredient_ids=np.unique(list(itertools.chain.from_iterable(list(PP_recipes_df["ingredient_ids_transformed"]))))
#ingredient_continuous_ids=pd.Series([i for i in range(1,len(all_ingredient_ids)+1)],index=list(np.sort(all_ingredient_ids)))
#ingredient_continuous_ids.to_pickle("./modules/dataframes/ingredient_continuous_ids.pkl")

In [ ]:
#ingredient_continuous_ids=pd.read_pickle("./modules/dataframes/ingredient_continuous_ids.pkl")
#n_recipes_ids=len(np.unique(PP_recipes_df["i"]))
#n_ingredients_ids=len(all_ingredient_ids)

#print(f"There are {n_recipes_ids} recipe ids")
#print(f"There are {n_ingredients_ids} ingredient ids")

There are 178265 recipe ids
There are 7993 ingredient ids


In [ ]:
#hash_encoder=EncodedHashedEmbeddings(n_recipes_ids,512,n_ingredients_ids,512)
#hashed_recipes_ids_encoded_embeddings,hashed_ingredients_ids_encoded_embeddings=hash_encoder.get_encoded_hashed_embeddings("./modules/hashed_encoded_tables")

In [11]:
hashed_ingredients_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_ingredients.pt")
hashed_recipes_ids_encoded_embeddings=torch.load("./modules/hashed_encoded_tables/hashed_embeddings_recipes.pt")

In [12]:
print(hashed_recipes_ids_encoded_embeddings.shape)
print(hashed_ingredients_ids_encoded_embeddings.shape)

torch.Size([178267, 512])
torch.Size([7995, 512])


In [13]:
#Loss with alpha = 0.3 and temperature = 0.2
loss_alpha=0.3
temperature=0.2
device=(torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))
hyperparemeters_ranges={
    "batch_size":[32, 64],
    "dropout":{
        "low":0.05,
        "high":0.3,
        "step":0.05
    },
    "projec_dropout":{
        "low":0.05,
        "high":0.3,
        "step":0.05
    },
    "lr":{
        "low":1e-4,
        "high":1e-2,
        "log":True
    },
    "weight_decay":{
        "low":1e-5,
        "high":1e-3,
        "log":True
    },
    "warmup_prop":{
        "low":0.01,
        "high":0.1,
        "step":0.01
    },
    "mean_mode": [True, False]
}
optuna_object=OptunaFunction(train_df=train_df_processed,val_df=val_df_processed,train_cls_embeddings=train_cls_embeddings,
                             val_cls_embeddings=val_cls_embeddings,train_mean_embeddings=train_mean_embeddings,
                             val_mean_embeddings=val_mean_embeddings,hashed_ingredients_ids_encoded_embeddings=hashed_ingredients_ids_encoded_embeddings,
                             hashed_recipes_ids_encoded_embeddings=hashed_recipes_ids_encoded_embeddings,loss_alpha=loss_alpha,temperature=temperature,
                             device=device,hyperparemeters_ranges=hyperparemeters_ranges)

In [15]:
study=optuna.create_study(direction="minimize",pruner=optuna.pruners.MedianPruner(),storage="sqlite:///optuna_study.db",load_if_exists=True,study_name="recommendation_system_model_study")

[I 2026-04-18 13:18:22,586] Using an existing study with name 'recommendation_system_model_study' instead of creating a new one.


In [16]:
study.optimize(optuna_object.objective,n_trials=10)

2026-04-18 13:18:40.059 | INFO     | modules.optuna_function:objective:85 - Trial number 0: 
2026-04-18 13:18:40.060 | INFO     | modules.optuna_function:objective:86 - batch_size:64_dropout:0,3_projec_dropout:0,25_lr:0,008834724109738035_weight_decay:0,0006779624554541989_warmup_prop:0,01_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-18 13:18:40.061 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-18 13:19:49.040 | INFO     | modules.train:run_training:178 - Epoch 0 : train total loss = 1.1700043656410428
2026-04-18 13:19:49.041 | INFO     | modules.train:run_training:179 - Epoch 0 : train contrastive loss = 3.8111283286995845
2026-04-18 13:19:49.042 | INFO     | modules.train:run_training:180 - Epoch 0 : train MSE loss = 0.03809402538292588
2026-04-18 13:19:49.043 | INFO     | modules.train:run_training:182 - Epoch 0 : validation total loss = 1.1644546907881033
2026-04-18 13:19:49.044 | INFO     | modules.train:run_training:183 - Epoch 0 : validation contrastive 

----------------------------------------------------------------------------------------------------


[I 2026-04-18 13:26:14,303] Trial 0 finished with value: 1.1467548168223838 and parameters: {'batch_size': 64, 'dropout': 0.3, 'projec_dropout': 0.25, 'lr': 0.008834724109738035, 'weight_decay': 0.0006779624554541989, 'warmup_prop': 0.01, 'mean_mode': True}. Best is trial 0 with value: 1.1467548168223838.
2026-04-18 13:26:14.868 | INFO     | modules.optuna_function:objective:85 - Trial number 1: 
2026-04-18 13:26:14.869 | INFO     | modules.optuna_function:objective:86 - batch_size:32_dropout:0,3_projec_dropout:0,2_lr:0,006752329720101161_weight_decay:5,195328204771679e-05_warmup_prop:0,04_mean_mode:False_loss_temp:0,2_loss_alpha:0,3
2026-04-18 13:26:14.870 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-18 13:27:25.164 | INFO     | modules.train:run_training:178 - Epoch 0 : train total loss = 0.9778979015568574
2026-04-18 13:27:25.165 | INFO     | modules.train:run_training:179 - Epoch 0 : train contrastive loss = 3.1686162583069617
2026-04-18 13:27:25.166 | INFO     |

----------------------------------------------------------------------------------------------------


[I 2026-04-18 13:32:33,051] Trial 1 finished with value: 0.9568283109254735 and parameters: {'batch_size': 32, 'dropout': 0.3, 'projec_dropout': 0.2, 'lr': 0.006752329720101161, 'weight_decay': 5.195328204771679e-05, 'warmup_prop': 0.04, 'mean_mode': False}. Best is trial 1 with value: 0.9568283109254735.
2026-04-18 13:32:33.634 | INFO     | modules.optuna_function:objective:85 - Trial number 2: 
2026-04-18 13:32:33.635 | INFO     | modules.optuna_function:objective:86 - batch_size:64_dropout:0,05_projec_dropout:0,1_lr:0,0001634316560750821_weight_decay:1,7058847746216797e-05_warmup_prop:0,03_mean_mode:False_loss_temp:0,2_loss_alpha:0,3
2026-04-18 13:32:33.636 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-18 13:33:43.715 | INFO     | modules.train:run_training:178 - Epoch 0 : train total loss = 1.1715094561970563
2026-04-18 13:33:43.716 | INFO     | modules.train:run_training:179 - Epoch 0 : train contrastive loss = 3.794558388377548
2026-04-18 13:33:43.718 | INFO    

...

[I 2026-04-18 13:55:00,871] Trial 4 finished with value: 1.1448075926822165 and parameters: {'batch_size': 64, 'dropout': 0.3, 'projec_dropout': 0.2, 'lr': 0.00017731617223446755, 'weight_decay': 6.669861714210607e-05, 'warmup_prop': 0.060000000000000005, 'mean_mode': False}. Best is trial 3 with value: 0.9517513487928657.
2026-04-18 13:55:01.449 | INFO     | modules.optuna_function:objective:85 - Trial number 5: 
2026-04-18 13:55:01.450 | INFO     | modules.optuna_function:objective:86 - batch_size:32_dropout:0,1_projec_dropout:0,3_lr:0,0006418734880444401_weight_decay:0,0001681295795906461_warmup_prop:0,06999999999999999_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-18 13:55:01.452 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-18 13:56:11.100 | INFO     | modules.train:run_training:178 - Epoch 0 : train total loss = 0.9805423834231134
2026-04-18 13:56:11.101 | INFO     | modules.train:run_training:179 - Epoch 0 : train contrastive loss = 3.161586904962221
2026

----------------------------------------------------------------------------------------------------


[I 2026-04-18 14:01:11,702] Trial 5 finished with value: 0.9465762235785043 and parameters: {'batch_size': 32, 'dropout': 0.1, 'projec_dropout': 0.3, 'lr': 0.0006418734880444401, 'weight_decay': 0.0001681295795906461, 'warmup_prop': 0.06999999999999999, 'mean_mode': True}. Best is trial 5 with value: 0.9465762235785043.
2026-04-18 14:01:12.436 | INFO     | modules.optuna_function:objective:85 - Trial number 6: 
2026-04-18 14:01:12.437 | INFO     | modules.optuna_function:objective:86 - batch_size:64_dropout:0,05_projec_dropout:0,3_lr:0,005817200547429063_weight_decay:1,3477437675661758e-05_warmup_prop:0,08_mean_mode:False_loss_temp:0,2_loss_alpha:0,3
2026-04-18 14:01:12.440 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-18 14:02:23.953 | INFO     | modules.train:run_training:178 - Epoch 0 : train total loss = 1.1680312878494963
2026-04-18 14:02:23.954 | INFO     | modules.train:run_training:179 - Epoch 0 : train contrastive loss = 3.797605397504404
2026-04-18 14:02:23.

----------------------------------------------------------------------------------------------------


[I 2026-04-18 14:08:51,703] Trial 7 finished with value: 0.956344969810978 and parameters: {'batch_size': 32, 'dropout': 0.2, 'projec_dropout': 0.2, 'lr': 0.0013844812885690293, 'weight_decay': 1.65014939934253e-05, 'warmup_prop': 0.04, 'mean_mode': False}. Best is trial 5 with value: 0.9465762235785043.
2026-04-18 14:08:52.449 | INFO     | modules.optuna_function:objective:85 - Trial number 8: 
2026-04-18 14:08:52.450 | INFO     | modules.optuna_function:objective:86 - batch_size:64_dropout:0,1_projec_dropout:0,3_lr:0,001877714943251037_weight_decay:2,3227782711772806e-05_warmup_prop:0,02_mean_mode:True_loss_temp:0,2_loss_alpha:0,3
2026-04-18 14:08:52.451 | INFO     | modules.train:run_training:89 - Epoch 0 : 
2026-04-18 14:10:04.484 | INFO     | modules.train:run_training:178 - Epoch 0 : train total loss = 1.1683571437083253
2026-04-18 14:10:04.485 | INFO     | modules.train:run_training:179 - Epoch 0 : train contrastive loss = 3.802050120239958
2026-04-18 14:10:04.486 | INFO     | m

In [ ]:
study_loaded=optuna.load_study(study_name="recommendation_system_model_study",storage="sqlite:///optuna_study.db")

In [ ]:
"""
study_best_components={
    "study_best_params":study_loaded.best_params,
    "study_best_value":study_loaded.best_value
}
torch.save(study_best_components,"./study_best_components.pt")
"""


In [19]:
fig=optuna.visualization.plot_param_importances(study_loaded)
fig.show()

In [20]:
fig=optuna.visualization.plot_parallel_coordinate(study_loaded)
fig.show()

In [21]:
fig=optuna.visualization.plot_optimization_history(study_loaded)
fig.show()

In [ ]:
"""
best_model_name="Best_Recommendation_System_model"
best_model_uri=f"runs:/7edda145087b428f97b3f753e1301f0b/model_5"
registration=mlflow.register_model(best_model_uri,best_model_name)
"""

c:\Users\sh032\anaconda3\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning:

The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.

Successfully registered model 'Best_Recommendation_System_model'.
2026/04/19 19:46:39 WARNING mlflow.tracking._model_registry.fluent: Run with id 7edda145087b428f97b3f753e1301f0b has no artifacts at artifact path 'model_5', registering model based on models:/m-d2b1b983ece842189a76e5eb1fce4715 instead
Created version '1' of model 'Best_Recommendation_System_model'.


In [ ]:
"""
client.transition_model_version_stage(
    name="Best_Recommendation_System_model",
    version=1,
    stage="Staging"
)
"""

C:\Users\sh032\AppData\Local\Temp\ipykernel_20180\3505277116.py:1: FutureWarning:

``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages



<ModelVersion: aliases=[], creation_timestamp=1776642399527, current_stage='Staging', deployment_job_state=None, description=None, last_updated_timestamp=1776642427017, metrics=[<Metric: dataset_digest=None, dataset_name=None, key='best_validation_contrastive_loss', model_id='m-d2b1b983ece842189a76e5eb1fce4715', run_id='7edda145087b428f97b3f753e1301f0b', step=0, timestamp=1776520843130, value=3.0634035935965915>,
 <Metric: dataset_digest=None, dataset_name=None, key='best_validation_MSE_loss', model_id='m-d2b1b983ece842189a76e5eb1fce4715', run_id='7edda145087b428f97b3f753e1301f0b', step=0, timestamp=1776520843130, value=0.03936444214415005>,
 <Metric: dataset_digest=None, dataset_name=None, key='best_validation_total_loss', model_id='m-d2b1b983ece842189a76e5eb1fce4715', run_id='7edda145087b428f97b3f753e1301f0b', step=0, timestamp=1776520843130, value=0.9465762235785043>,
 <Metric: dataset_digest=None, dataset_name=None, key='epoch_train_contrastive_loss', model_id='m-d2b1b983ece842189a

In [ ]:
"""
model_version=client.get_model_version("Best_Recommendation_System_model",1)
version=int(model_version.version)
client.set_model_version_tag(name="Best_Recommendation_System_model",key="Role",value="Challenger",version=version)
client.set_model_version_tag(name="Best_Recommendation_System_model",key="Stage",value="Staging",version=version)
client.set_registered_model_alias(name="Best_Recommendation_System_model",alias="Challenger",version=version)
"""

In [6]:
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

model_version=client.get_model_version("Best_Recommendation_System_model",1)
best_model_registered_uri=f"models:/Best_Recommendation_System_model/{model_version.version}"
best_recommendation_model=mlflow.pytorch.load_model(best_model_registered_uri,map_location=torch.device("cpu"))
best_recommendation_model.device=device


batch_size=int(model_version.params["batch_size"])
loss_temperature=float(model_version.params["loss_temperature"])
loss_alpha=float(model_version.params["loss_alpha"])
run_id=model_version.run_id

test_dataset=CreationDataset(test_df_processed,test_cls_embeddings,test_mean_embeddings)
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunction()
collate_fn=collate_function_object.collate_fn
test_dataloader=DataLoader(test_dataset,batch_size=batch_size,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=True,drop_last=True)

tester=Test(test_dataloader=test_dataloader,model=best_recommendation_model,device=device,loss_alpha=loss_alpha,loss_temperature=loss_temperature)
test_total_loss,test_contrastive_loss,test_mse_loss=tester.run_testing(f"./test_savings",run_id=run_id)

c:\Users\sh032\anaconda3\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning:

The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.

2026/04/22 19:03:04 WARNING mlflow.pytorch: Stored model version '2.10.0+cu128' does not match installed PyTorch version '2.8.0+cpu'
2026-04-22 19:03:05.332 | INFO     | modules.test:run_testing:25 - Start of the test :
2026-04-22 19:05:05.426 | INFO     | modules.test:run_testing:63 - Test Total loss = 0.950422175468937
2026-04-22 19:05:05.427 | INFO     | modules.test:run_testing:64 - Test Contrastive loss = 3.0739834693170365
2026-04-22 19:05:05.427 | INFO     | modules.test:run_testing:65 - Test MSE loss = 0.04032442552508206


In [ ]:
"""
model_version=client.get_model_version("Best_Recommendation_System_model",1)
version=int(model_version.version)
test_metrics=torch.load("./test_savings/test_metrics.pt",weights_only=False)
for key,value in test_metrics.items():
    client.set_model_version_tag("Best_Recommendation_System_model",version=1,key=key,value=value)
"""

In [7]:
recipes_df,recipes_set_cls_embeddings,recipes_set_mean_embeddings=creation_recipes_set(full_df,all_cls_embeddings,all_mean_embeddings,path="./modules/recipes_set")

In [8]:
print(recipes_df.shape)
print(recipes_set_cls_embeddings.shape)
print(recipes_set_mean_embeddings.shape)

(15423, 13)
torch.Size([15423, 768])
torch.Size([15423, 768])


In [9]:
PP_recipes_df=pd.read_csv("data/PP_recipes.csv")
PP_recipes_df["len_ingredients"]=PP_recipes_df["ingredient_ids"].str.count(r",")+1
max_len_ingredients=PP_recipes_df["len_ingredients"].max()
PP_recipes_df.drop(columns=["len_ingredients"],inplace=True)

In [10]:
recipe_before_split_processor=BeforeSplitPreprocessingRecipesFeatures(recipes_df,max_len_ingredients)
recipes_df_processed=recipe_before_split_processor.preprocessing()

In [11]:
ingredient_continuous_ids=pd.read_pickle("./modules/dataframes/ingredient_continuous_ids.pkl")

In [12]:
def mapping_list(x:list,mapping_serie:pd.Series):
    return [mapping_serie.loc[i] if i in mapping_serie.index else 0 for i in x]
recipes_df_processed["ingredient_ids_continuous"]=recipes_df_processed["ingredient_ids"].apply(lambda x : mapping_list(x,ingredient_continuous_ids))

In [13]:
recipes_df_processed["full_text"]=recipes_df_processed["name"]+" [SEP] "+recipes_df_processed["description"]+" [SEP] "+recipes_df_processed["steps"]+" [SEP] "+recipes_df_processed["tags"]

In [14]:
recipe_after_split_processor=AfterSplitPreprocessingRecipesFeatures(train_df)
recipes_df_processed=recipe_after_split_processor.preprocessing(recipes_df_processed)

In [15]:
scaler_recipes=joblib.load("./modules/scalers/scaler_recipes.pkl")
recipes_scaler=ScalerInferenceRecipes(scaler_recipes)
recipes_df_processed=recipes_scaler.scale(recipes_df_processed)

In [16]:
recipes_df_processed.to_parquet("./modules/recipes_set/recipes_df_processed.parquet")

In [6]:
recipes_processed_parquet_file=pq.ParquetFile("./modules/recipes_set/recipes_df_processed.parquet")
recipes_parquet_file=pq.ParquetFile("./modules/recipes_set/recipes_df.parquet")

recipes_df=load(recipes_parquet_file)
recipes_df_processed=load(recipes_processed_parquet_file)
recipes_set_cls_embeddings=torch.load("./modules/recipes_set/recipes_set_cls_embeddings.pt")
recipes_set_mean_embeddings=torch.load("./modules/recipes_set/recipes_set_mean_embeddings.pt")

In [7]:
print(recipes_df.columns)

Index(['recipe_id', 'i', 'name', 'description', 'tags', 'calorie_level',
       'minutes', 'n_ingredients', 'ingredient_ids', 'nutrition', 'n_steps',
       'steps', 'techniques_recipes'],
      dtype='object')


In [8]:
print(recipes_df_processed.columns)
print(recipes_df_processed.shape)
print(recipes_set_cls_embeddings.shape)
print(recipes_set_mean_embeddings.shape)


Index(['recipe_id', 'i', 'name', 'description', 'tags', 'calorie_level',
       'minutes', 'n_ingredients', 'ingredient_ids', 'nutrition', 'n_steps',
       'steps', 'techniques_recipes', 'ingredient_ids_continuous', 'full_text',
       'calorie_level_scaled', 'minutes_scaled', 'n_steps_scaled',
       'n_ingredients_scaled'],
      dtype='object')
(15423, 19)
torch.Size([15423, 768])
torch.Size([15423, 768])


In [9]:
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

model_version=client.get_model_version("Best_Recommendation_System_model",1)
best_model_registered_uri=f"models:/Best_Recommendation_System_model/1"
best_recommendation_model=mlflow.pytorch.load_model(best_model_registered_uri,map_location=torch.device("cpu"))
best_recommendation_model.device=device

batch_size=int(model_version.params["batch_size"])
recipes_dataset=CreationDataset(recipes_df_processed,recipes_set_cls_embeddings,recipes_set_mean_embeddings)
generator=torch.Generator()
generator.manual_seed(seed)
collate_function_object=CollateFunctionInferenceRecipes()
collate_fn=collate_function_object.collate_fn
recipes_dataloader=DataLoader(recipes_dataset,batch_size=batch_size,collate_fn=collate_fn,generator=generator,worker_init_fn=seed_worker,shuffle=False,drop_last=False)


recipes_embeddings_extractor=RecipesEmbeddingsExtractor(model=best_recommendation_model,device=device)
recipes_embeddings=recipes_embeddings_extractor.get_recipes_embeddings(dataloader=recipes_dataloader,path=f"./modules/recipes_set")

c:\Users\sh032\anaconda3\Lib\site-packages\mlflow\tracking\_model_registry\utils.py:220: FutureWarning:

The filesystem model registry backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.

2026/04/22 19:09:11 WARNING mlflow.pytorch: Stored model version '2.10.0+cu128' does not match installed PyTorch version '2.8.0+cpu'


In [9]:
recipes_embeddings=torch.load("./modules/recipes_set/recipes_embeddings.pt")
print(recipes_embeddings.shape)

torch.Size([15423, 256])


In [7]:
recipes_parquet_file=pq.ParquetFile("./modules/recipes_set/recipes_df.parquet")
recipes_df=load(recipes_parquet_file)

recipes_embeddings=torch.load(f"./modules/recipes_set/recipes_embeddings.pt")

scaler_user=joblib.load("./modules/scalers/scaler_users.pkl")

PP_users_df["len_items"]=PP_users_df["items"].str.count(r",")+1
max_len_items=PP_users_df["len_items"].max()
PP_users_df.drop(columns=["len_items"],inplace=True)

n_techniques_users=len(full_df_processed.loc[~full_df_processed["techniques_users"].isna(),"techniques_users"].iloc[0])

print(max_len_items)
print(n_techniques_users)



6437
58


In [ ]:
device = (torch.device("mps") if torch.backends.mps.is_available() else torch.device("cuda" if torch.cuda.is_available() else "cpu"))

best_model_registered_uri=f"models:/Best_Recommendation_System_model/1"
best_recommendation_model=mlflow.pytorch.load_model(best_model_registered_uri,map_location=torch.device("cpu"))
best_recommendation_model.device=device

recipes_recommender=RecipesRecommender(device=device,model=best_recommendation_model,recipes_embeddings=recipes_embeddings,recipes_df=recipes_df,train_df=train_df,scaler_user=scaler_user,max_len_items=max_len_items,n_techniques_users=n_techniques_users)

2026/04/22 20:52:35 WARNING mlflow.pytorch: Stored model version '2.10.0+cu128' does not match installed PyTorch version '2.8.0+cpu'


In [12]:
user_id=full_df.loc[full_df["rating"]==5,"user_id"].iloc[0]
ratings=full_df.loc[full_df["rating"]==5,"ratings"].iloc[0]
items=full_df.loc[full_df["rating"]==5,"items"].iloc[0]
techniques_users=full_df.loc[full_df["rating"]==5,"techniques_users"].iloc[0]
n_items=full_df.loc[full_df["rating"]==5,"n_items"].iloc[0]
n_ratings=full_df.loc[full_df["rating"]==5,"n_ratings"].iloc[0]
n_recommended_recipes=5

recommended_recipes=recipes_recommender.get_recommendations(
    user_id=user_id,
    ratings=ratings,
    items=items,
    techniques_users=techniques_users,
    n_items=n_items,
    n_ratings=n_ratings,
    n_recommended_recipes=n_recommended_recipes
)

c:\Users\sh032\anaconda3\Lib\site-packages\sklearn\base.py:464: UserWarning:

X does not have valid feature names, but StandardScaler was fitted with feature names



In [13]:
display(recommended_recipes)

,recipe_id,i,name,description,tags,calorie_level,minutes,n_ingredients,ingredient_ids,nutrition,n_steps,steps,techniques_recipes,cos_sim_scores
10256,70046,117001,white pizza with shellfish,this is a quick dinner solution for the weekda...,"['60-minutes-or-less', 'time-to-make', 'course...",0,35,9,"[5483, 5006, 4814, 3184, 6336, 6487, 7213, 344...","[185.8, 21.0, 7.0, 7.0, 22.0, 30.0, 1.0]",7,"['heat oven to 400', 'place pizza crust on a g...","[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.952273
3600,63624,40832,the best darn way to barbecue chicken,this chicken is basted with a buttery herb sau...,"['weeknight', 'time-to-make', 'course', 'main-...",2,70,11,"[1240, 4574, 2325, 3184, 6183, 7168, 5106, 382...","[882.0, 129.0, 1.0, 34.0, 59.0, 86.0, 0.0]",11,['prepare the chicken as indicated then wash i...,"[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.949260
13124,100353,150519,baked spaghetti casserole,we love the market day spaghetti pie casserole...,"['60-minutes-or-less', 'time-to-make', 'course...",2,45,13,"[132, 5180, 2499, 4717, 4574, 3484, 3184, 5010...","[4002.7, 278.0, 65.0, 81.0, 443.0, 321.0, 118.0]",16,['preheat oven to 350 and pam a 13x9 deep baki...,"[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, ...",0.948888
11753,211537,134346,beijing chicken and dumplings,"ah, the comforting feeling of a big pot of chi...","['time-to-make', 'course', 'main-ingredient', ...",0,100,15,"[1240, 1257, 5975, 3269, 1251, 3184, 6696, 762...","[240.5, 23.0, 15.0, 25.0, 37.0, 21.0, 2.0]",7,"['in a large soup pot , combine all ingredient...","[0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0.945110
5916,93317,67152,savory chicken egg rolls with sweet and sour ...,"a fun change from ordinary egg rolls, this chi...","['time-to-make', 'course', 'main-ingredient', ...",0,65,18,"[3492, 2499, 1706, 8021, 6335, 1093, 893, 3046...","[162.0, 4.0, 15.0, 8.0, 15.0, 3.0, 8.0]",9,['combine all filling ingredients in a large b...,"[0, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, ...",0.944351
